# Week 4: Ablation Study
**Project**: Quantum Kernels on CLIP/BERT Embeddings

**All experiments run at N=500 to match Week 3.**

**Ablations:**
1. Feature maps — Angle-only vs IQP 1-layer vs IQP 2-layer (8 qubits)
2. Qubit count — 4 vs 8 qubits (IQP 1-layer)
3. C parameter — sweep at N=200 to find best value, then apply at N=500
4. Multi-seed — 5 seeds for statistical robustness (optional, ~100 min)

**Week 3 result to build on:**
- Quantum IQP 1-layer, 8 qubits, N=500: Acc=0.6500, F1=0.7107
- Best classical: Linear at Acc=0.6700, F1=0.7080

**Runtime:** ~65 min without multi-seed | ~165 min with multi-seed

In [ ]:
!pip install pennylane pennylane-lightning --quiet

import pennylane as qml
import numpy as np
import time, json, pickle, warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')

print(f"PennyLane: {qml.__version__}")

dev4 = qml.device('lightning.qubit', wires=4)
dev8 = qml.device('lightning.qubit', wires=8)

---
## Step 1: Load all saved data

In [ ]:
# Upload: embeddings_week2.npz, baseline_results.pkl, kernel_matrices_week3.npz

data = np.load('embeddings_week2.npz')
wk3  = np.load('kernel_matrices_week3.npz')

X_tr_full = data['X_tr_subset'].astype(np.float64)   # (500, 8)
X_te_full = data['X_te_subset'].astype(np.float64)   # (100, 8)
y_tr_full = data['y_tr_subset']
y_te_full = data['y_te_subset']

# Week 3 saved kernel matrices (500x500 and 100x500 — matches N=500)
K_wk3_train = wk3['K_train_quantum']   # 500x500
K_wk3_test  = wk3['K_test_quantum']    # 100x500

with open('baseline_results.pkl', 'rb') as f:
    baselines = pickle.load(f)

N_ABL = 500   # all ablations at N=500
X_tr  = X_tr_full[:N_ABL]
y_tr  = y_tr_full[:N_ABL]
X_te  = X_te_full
y_te  = y_te_full

print(f"Training set : {X_tr.shape}")
print(f"Test set     : {X_te.shape}")
print(f"K_wk3_train  : {K_wk3_train.shape}  (IQP 1L, pre-computed)")
print(f"K_wk3_test   : {K_wk3_test.shape}")
print()
print("Classical baselines (N=500):")
for k, v in baselines.items():
    print(f"  {k:<14} acc={v['accuracy']:.4f}  f1={v['f1']:.4f}")

---
## Step 2: Define all three feature maps

In [ ]:
def angle_map(x, wires):
    """Angle embedding only — RY per qubit, no entanglement."""
    for i, w in enumerate(wires):
        qml.RY(x[i], wires=w)

def iqp_1layer(x, wires):
    """IQP 1-layer: Hadamard + RZ encoding + ZZ entanglement."""
    n = len(wires)
    qml.broadcast(qml.Hadamard, wires=wires, pattern='single')
    for i, w in enumerate(wires):
        qml.RZ(x[i], wires=w)
    for i in range(n - 1):
        qml.CNOT(wires=[wires[i], wires[i+1]])
        qml.RZ(x[i] * x[i+1], wires=wires[i+1])
        qml.CNOT(wires=[wires[i], wires[i+1]])

def iqp_2layer(x, wires):
    """IQP 2-layer: same block repeated twice for higher-order interactions."""
    for _ in range(2):
        iqp_1layer(x, wires)

def make_kernel(feature_map, device):
    @qml.qnode(device)
    def circuit(x1, x2):
        feature_map(x1, wires=range(device.num_wires))
        qml.adjoint(feature_map)(x2, wires=range(device.num_wires))
        return qml.probs(wires=range(device.num_wires))
    return lambda x1, x2: float(circuit(x1, x2)[0])

k_angle = make_kernel(angle_map,  dev8)
k_iqp1  = make_kernel(iqp_1layer, dev8)
k_iqp2  = make_kernel(iqp_2layer, dev8)
k_iqp1_4q = make_kernel(iqp_1layer, dev4)

# Sanity checks
x0, x1 = X_tr[0], X_tr[1]
print("Sanity checks (K(x,x) must be 1.0):")
for name, kfn in [('Angle', k_angle), ('IQP 1L', k_iqp1), ('IQP 2L', k_iqp2), ('IQP 1L (4q)', k_iqp1_4q)]:
    kxx = kfn(x0[:4] if '4q' in name else x0,
               x0[:4] if '4q' in name else x0)
    print(f"  {name:<14} K(x,x)={kxx:.6f}")

# Single eval timing
t0 = time.time()
for _ in range(3): k_iqp1(x0, x1)
ms = (time.time()-t0)/3*1000
print(f"\nSingle IQP 1L eval: {ms:.1f}ms")
print(f"N=500 kernel matrix estimate: {500*500*ms/1000/60:.0f} min")

---
## Ablation 1: Feature map comparison
**8 qubits, N=500 | ~40 min for 2 new maps**

IQP 1-layer is reused from the Week 3 saved kernel matrix — no recomputation.

In [ ]:
fm_results = {}

# ── IQP 1L: reuse Week 3 (free) ──
svm_wk3   = SVC(kernel='precomputed', C=1.0).fit(K_wk3_train, y_tr)
y_pred_wk3 = svm_wk3.predict(K_wk3_test)
fm_results['IQP 1L'] = {
    'accuracy': float(accuracy_score(y_te, y_pred_wk3)),
    'f1':       float(f1_score(y_te, y_pred_wk3)),
    'compute_s': 0.0, 'reused': True
}
print(f"IQP 1L (reused from Week 3): acc={fm_results['IQP 1L']['accuracy']:.4f}  f1={fm_results['IQP 1L']['f1']:.4f}")

# ── Angle and IQP 2L: compute fresh ──
for fname, kfn in [('Angle', k_angle), ('IQP 2L', k_iqp2)]:
    print(f"\nComputing {fname} kernel (N={N_ABL})...")
    t0 = time.time()
    Ktr = qml.kernels.kernel_matrix(X_tr, X_tr, kfn)
    Kte = qml.kernels.kernel_matrix(X_te, X_tr, kfn)
    elapsed = time.time() - t0
    y_p = SVC(kernel='precomputed', C=1.0).fit(Ktr, y_tr).predict(Kte)
    fm_results[fname] = {
        'accuracy': float(accuracy_score(y_te, y_p)),
        'f1':       float(f1_score(y_te, y_p)),
        'compute_s': elapsed,
        'K_train': Ktr, 'K_test': Kte
    }
    print(f"  Done in {elapsed/60:.1f}min | acc={fm_results[fname]['accuracy']:.4f}  f1={fm_results[fname]['f1']:.4f}")

print(f"\n{'Feature Map':<12} {'Accuracy':>10} {'F1':>8} {'Time':>10}")
print("-"*44)
for name, r in fm_results.items():
    t_s = 'reused' if r.get('reused') else f"{r['compute_s']/60:.1f}min"
    print(f"{name:<12} {r['accuracy']:>10.4f} {r['f1']:>8.4f} {t_s:>10}")

---
## Ablation 2: Qubit count
**IQP 1-layer, N=500 | ~20 min for 4-qubit run**

4 qubits uses the first 4 PCA dimensions (highest-variance components).  
8 qubits reuses the Week 3 result.  
12 qubits requires re-running Week 2 with `n_components=12`.

In [ ]:
qubit_results = {}

# 8 qubits — reuse Week 3
qubit_results[8] = {
    'accuracy': fm_results['IQP 1L']['accuracy'],
    'f1':       fm_results['IQP 1L']['f1'],
    'reused': True
}

# 4 qubits — first 4 PCA dims
X_tr_4 = X_tr[:, :4]
X_te_4 = X_te[:, :4]
print(f"Computing IQP 1L kernel (4 qubits, N={N_ABL})...")
t0 = time.time()
K4_tr = qml.kernels.kernel_matrix(X_tr_4, X_tr_4, k_iqp1_4q)
K4_te = qml.kernels.kernel_matrix(X_te_4, X_tr_4, k_iqp1_4q)
elapsed = time.time() - t0
y_p4 = SVC(kernel='precomputed', C=1.0).fit(K4_tr, y_tr).predict(K4_te)
qubit_results[4] = {
    'accuracy': float(accuracy_score(y_te, y_p4)),
    'f1':       float(f1_score(y_te, y_p4)),
    'compute_s': elapsed
}
print(f"  Done in {elapsed/60:.1f}min | acc={qubit_results[4]['accuracy']:.4f}  f1={qubit_results[4]['f1']:.4f}")

print(f"\n{'Qubits':>8} {'Accuracy':>10} {'F1':>8}")
print("-"*30)
for n_q in sorted(qubit_results.keys()):
    r = qubit_results[n_q]
    print(f"{n_q:>8} {r['accuracy']:>10.4f} {r['f1']:>8.4f}")
print(f"{'12':>8} {'(re-run Week 2 with n_components=12)':>10}")

---
## Ablation 3: SVM C parameter
**IQP 1-layer, 8 qubits | C sweep at N=200 (~15 min), best C applied to N=500**

We find the best C at N=200 (fast), then re-run once at N=500 with that value.  
This avoids a 5 × 20-min grid search.

In [ ]:
# ── Step 3a: C sweep at N=200 for speed ──
N_CSWEEP = 200
X_tr_c = X_tr[:N_CSWEEP]
y_tr_c = y_tr[:N_CSWEEP]

print(f"C sweep at N={N_CSWEEP} (heuristic to find best C)...")
t0 = time.time()
Kc_tr = qml.kernels.kernel_matrix(X_tr_c, X_tr_c, k_iqp1)
Kc_te = qml.kernels.kernel_matrix(X_te,   X_tr_c, k_iqp1)
print(f"  Kernel matrix done in {time.time()-t0:.0f}s")

C_values  = [0.1, 0.5, 1.0, 5.0, 10.0]
c_results = {}
print(f"\n{'C':>6}  {'Accuracy':>10}  {'F1':>8}")
print("-"*28)
for C in C_values:
    y_pc = SVC(kernel='precomputed', C=C).fit(Kc_tr, y_tr_c).predict(Kc_te)
    acc_c = accuracy_score(y_te, y_pc)
    f1_c  = f1_score(y_te, y_pc)
    c_results[C] = {'accuracy': acc_c, 'f1': f1_c}
    print(f"{C:>6.1f}  {acc_c:>10.4f}  {f1_c:>8.4f}")

best_C = max(c_results, key=lambda c: c_results[c]['accuracy'])
print(f"\nBest C = {best_C}")

# ── Step 3b: Re-run at N=500 with best C (skip if best_C == 1.0) ──
if best_C == 1.0:
    print(f"\nC=1.0 is already the default used in all ablations above.")
    print("No re-run needed — all existing results are optimal.")
    c_best_result = fm_results['IQP 1L']
else:
    print(f"\nRe-running IQP 1L at N=500 with C={best_C}...")
    svm_bc = SVC(kernel='precomputed', C=best_C).fit(K_wk3_train, y_tr)
    y_pbc  = svm_bc.predict(K_wk3_test)
    c_best_result = {
        'accuracy': float(accuracy_score(y_te, y_pbc)),
        'f1':       float(f1_score(y_te, y_pbc))
    }
    print(f"  C={best_C}: acc={c_best_result['accuracy']:.4f}  f1={c_best_result['f1']:.4f}")
    print(f"  C=1.0   : acc={fm_results['IQP 1L']['accuracy']:.4f}  f1={fm_results['IQP 1L']['f1']:.4f}")

---
## Ablation 4: Multi-seed robustness
**IQP 1-layer, 8 qubits, N=500 | 5 seeds × ~20 min ≈ 100 min**

**This section is optional.** Run it before paper submission for statistical credibility.  
Each seed draws a different random subset of 500 training samples.

In [ ]:
RUN_MULTISEED = False   # ← set to True when you have time (~100 min)

seed_accs, seed_f1s = [], []

if RUN_MULTISEED:
    N_SEEDS = 5
    print(f"Multi-seed: {N_SEEDS} seeds × N={N_ABL}")
    print(f"{'Seed':>6}  {'Accuracy':>10}  {'F1':>8}  {'Time':>8}")
    print("-"*38)
    for seed in range(N_SEEDS):
        rng = np.random.RandomState(seed)
        idx = rng.choice(len(X_tr_full), N_ABL, replace=False)
        Xs, ys = X_tr_full[idx], y_tr_full[idx]
        t0 = time.time()
        Ks_tr = qml.kernels.kernel_matrix(Xs,   Xs,   k_iqp1)
        Ks_te = qml.kernels.kernel_matrix(X_te, Xs,   k_iqp1)
        y_ps  = SVC(kernel='precomputed', C=best_C).fit(Ks_tr, ys).predict(Ks_te)
        acc_s, f1_s = accuracy_score(y_te, y_ps), f1_score(y_te, y_ps)
        seed_accs.append(acc_s); seed_f1s.append(f1_s)
        print(f"{seed:>6}  {acc_s:>10.4f}  {f1_s:>8.4f}  {time.time()-t0:>6.0f}s")
    print(f"\nMean acc: {np.mean(seed_accs):.4f} ± {np.std(seed_accs):.4f}")
    print(f"Mean F1 : {np.mean(seed_f1s):.4f} ± {np.std(seed_f1s):.4f}")
else:
    print("Multi-seed skipped (RUN_MULTISEED=False).")
    print("Set RUN_MULTISEED=True and re-run this cell when ready.")

---
## Paper Table 1

In [ ]:
print("=" * 70)
print(f"TABLE 1 — Quantum vs Classical SVM on 8-dim BERT Embeddings (SST-2)")
print("=" * 70)
print(f"{'Method':<26} {'Qubits':>7} {'N':>5} {'Acc':>8} {'F1':>8} {'vs Linear':>10}")
print("-" * 70)

# Classical
for name, vals in baselines.items():
    gap = vals['accuracy'] - 0.6700
    print(f"  {'Classical — '+name:<24} {'—':>7} {'500':>5} "
          f"{vals['accuracy']:>8.4f} {vals['f1']:>8.4f} {gap:>+10.1%}")
print()

# Feature map ablation
for fname, r in fm_results.items():
    gap  = r['accuracy'] - 0.6700
    note = ' *' if fname == 'IQP 1L' else ''
    print(f"  {'Quantum — '+fname:<24} {'8':>7} {'500':>5} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {gap:>+10.1%}{note}")
print()

# Qubit ablation
for n_q in sorted(qubit_results.keys()):
    r   = qubit_results[n_q]
    gap = r['accuracy'] - 0.6700
    print(f"  {'Quantum — IQP 1L':<24} {n_q:>7} {'500':>5} "
          f"{r['accuracy']:>8.4f} {r['f1']:>8.4f} {gap:>+10.1%}")
print(f"  {'Quantum — IQP 1L':<24} {'12':>7} {'—':>5} "
      f"{'TBD':>8} {'TBD':>8} {'(needs Wk2 re-run)':>12}")
print()

# Multi-seed
if seed_accs:
    gap = np.mean(seed_accs) - 0.6700
    print(f"  {'Quantum — IQP 1L (5-seed)':<24} {'8':>7} {'500':>5} "
          f"{np.mean(seed_accs):>8.4f} {np.mean(seed_f1s):>8.4f} {gap:>+10.1%}"
          f"  ±{np.std(seed_accs):.4f}")

print("=" * 70)
print("* IQP 1L result reused from Week 3 saved kernel matrix")

---
## Figures

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ── Fig 1: Feature map ──
fm_names  = list(fm_results.keys())
fm_accs   = [fm_results[n]['accuracy'] for n in fm_names]
fm_f1s    = [fm_results[n]['f1']       for n in fm_names]
colors_fm = ['#4D9DE0', '#E15554', '#3BB273']

axes[0].bar(fm_names, fm_accs, color=colors_fm, width=0.5, edgecolor='white', linewidth=1.5)
axes[0].axhline(0.6700, color='gray', linestyle='--', alpha=0.6, label='Linear SVM (best classical)')
axes[0].set_ylim(0.5, 0.80)
axes[0].set_title(f'Feature map comparison\n(8 qubits, N={N_ABL})', fontsize=11)
axes[0].set_ylabel('Accuracy'); axes[0].legend(fontsize=9)
for bar, val in zip(axes[0].patches, fm_accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.004, f'{val:.3f}',
                 ha='center', fontsize=11, fontweight='500')

# ── Fig 2: Qubit count ──
q_labels = [f'{n}q' for n in sorted(qubit_results)] + ['12q*']
q_accs   = [qubit_results[n]['accuracy'] for n in sorted(qubit_results)] + [None]
bar_cols = ['#4D9DE0', '#E15554', '#CCCCCC']
bars2 = axes[1].bar(q_labels, [v if v else 0 for v in q_accs],
                     color=bar_cols, width=0.5, edgecolor='white', linewidth=1.5)
axes[1].axhline(0.6700, color='gray', linestyle='--', alpha=0.6, label='Linear SVM baseline')
axes[1].set_ylim(0.5, 0.80)
axes[1].set_title(f'Qubit count comparison\n(IQP 1L, N={N_ABL})', fontsize=11)
axes[1].set_ylabel('Accuracy'); axes[1].legend(fontsize=9)
for bar, val in zip(bars2, q_accs):
    lbl = f'{val:.3f}' if val else 'TBD'
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                 lbl, ha='center', fontsize=11, fontweight='500')
axes[1].text(bars2[-1].get_x()+bars2[-1].get_width()/2, 0.52,
             '*needs\nWk2 re-run', ha='center', fontsize=8, color='#888')

# ── Fig 3: C sweep ──
C_vals  = list(c_results.keys())
c_accs_plot = [c_results[c]['accuracy'] for c in C_vals]
best_idx = c_accs_plot.index(max(c_accs_plot))
bar_cols3 = ['#E15554' if i==best_idx else '#4D9DE0' for i in range(len(C_vals))]
axes[2].bar([str(c) for c in C_vals], c_accs_plot, color=bar_cols3,
             width=0.6, edgecolor='white', linewidth=1.5)
axes[2].axhline(0.6700, color='gray', linestyle='--', alpha=0.6, label='Linear SVM baseline')
axes[2].set_ylim(0.5, 0.80)
axes[2].set_title(f'C parameter sweep\n(IQP 1L, 8q, N=200 heuristic)', fontsize=11)
axes[2].set_xlabel('C value'); axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=9)
for bar, val in zip(axes[2].patches, c_accs_plot):
    axes[2].text(bar.get_x()+bar.get_width()/2, val+0.004, f'{val:.3f}',
                 ha='center', fontsize=11, fontweight='500')

plt.suptitle(f'Week 4 Ablation Study — N={N_ABL}, SST-2, 8-dim BERT Embeddings',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('fig3_ablations.pdf', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fig3_ablations.pdf")

---
## Save results

In [ ]:
all_results = {
    'N_ABL': N_ABL,
    'feature_map_ablation': {k: {'accuracy': v['accuracy'], 'f1': v['f1']}
                              for k, v in fm_results.items()},
    'qubit_ablation':       {str(k): {'accuracy': v['accuracy'], 'f1': v['f1']}
                              for k, v in qubit_results.items()},
    'c_sweep':              {str(k): {'accuracy': v['accuracy'], 'f1': v['f1']}
                              for k, v in c_results.items()},
    'best_C':               float(best_C),
    'multi_seed': {
        'ran': bool(seed_accs),
        'mean_accuracy': float(np.mean(seed_accs)) if seed_accs else None,
        'std_accuracy':  float(np.std(seed_accs))  if seed_accs else None,
        'mean_f1':       float(np.mean(seed_f1s))  if seed_f1s  else None,
        'std_f1':        float(np.std(seed_f1s))   if seed_f1s  else None,
    },
    'classical_baselines': {k: {'accuracy': v['accuracy'], 'f1': v['f1']}
                             for k, v in baselines.items()},
}

with open('results_week4.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Saved: results_week4.json")
print()
print("Files to carry into Week 5:")
print("  results_week4.json          — all ablation numbers")
print("  results_week3.json          — learning curve data")
print("  kernel_matrices_week3.npz   — for kernel alignment analysis")
print("  embeddings_week2.npz        — for t-SNE visualisation")